In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder


In [ ]:
df = pd.read_csv('Agriculture_price_dataset.csv')


In [ ]:
df.sample(10)

,STATE,District Name,Market Name,Commodity,Variety,Grade,Min_Price,Max_Price,Modal_Price,Price Date
474245,Uttar Pradesh,bijnor,Najibabad,Onion,Red,FAQ,2000.0,2400.0,2200.0,9/11/2024
631453,Madhya Pradesh,ujjain,Ujjain,Onion,Onion,FAQ,400.0,1366.0,1200.0,3/12/2025
439569,Maharashtra,ahmednagar,Shree Sairaj Krushi Market,Onion,Local,FAQ,1500.0,3100.0,2300.0,8/7/2024
727338,Madhya Pradesh,dhar,Badnawar,Onion,Onion,FAQ,200.0,1100.0,200.0,6/2/2025
314812,Orissa,rayagada,Rayagada,Potato,Other,FAQ,1700.0,1800.0,1800.0,2/26/2024
598977,West Bengal,nadia,Karimpur,Potato,Jyoti,FAQ,1400.0,1500.0,1450.0,2/12/2025
247852,West Bengal,jalpaiguri,Belacoba,Potato,Jyoti,FAQ,1000.0,1200.0,1100.0,12/18/2023
662382,Punjab,muktsar,Muktsar,Onion,Nasik,FAQ,1500.0,2000.0,1800.0,4/10/2025
319486,Chandigarh,chandigarh,Chandigarh(Grain/Fruit),Potato,Other,FAQ,500.0,1000.0,800.0,3/2/2024
607260,West Bengal,nadia,Bethuadahari,Potato,F.A.Q.,Non-FAQ,800.0,900.0,840.0,2/19/2025


In [ ]:

df.duplicated().sum()

np.int64(0)

--- MANAGE DATE TIME ---

In [ ]:
df['Price Date'] = pd.to_datetime(df['Price Date'])

df['Year'] = df['Price Date'].dt.year
df['Month'] = df['Price Date'].dt.month
df['Day'] = df['Price Date'].dt.day 

df = df.drop('Price Date' , axis=1)

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 737392 entries, 0 to 737391
Data columns (total 12 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   STATE          737392 non-null  str    
 1   District Name  737392 non-null  str    
 2   Market Name    737392 non-null  str    
 3   Commodity      737392 non-null  str    
 4   Variety        737392 non-null  str    
 5   Grade          737392 non-null  str    
 6   Min_Price      737392 non-null  float64
 7   Max_Price      737392 non-null  float64
 8   Modal_Price    737392 non-null  float64
 9   Year           737392 non-null  int32  
 10  Month          737392 non-null  int32  
 11  Day            737392 non-null  int32  
dtypes: float64(3), int32(3), str(6)
memory usage: 59.1 MB


--- ONE HOT ENCODING ---

In [ ]:
df = pd.get_dummies(df , columns=['STATE' , 'Commodity' , 'Grade'] , drop_first=True , dtype="int8")

In [ ]:
df.head()

,District Name,Market Name,Variety,Min_Price,Max_Price,Modal_Price,Year,Month,Day,STATE_Andhra Pradesh,...,Commodity_Potato,Commodity_Rice,Commodity_Tomato,Commodity_Wheat,Grade_Large,Grade_Local,Grade_Medium,Grade_Non-FAQ,Grade_Ref grade-1,Grade_Ref grade-2
0,nashik,Lasalgaon(Niphad),Maharashtra 2189,2172.0,2399.0,2300.0,2023,6,6,0,...,0,0,0,1,0,0,0,0,0,0
1,satara,Patan,Other,1000.0,1500.0,1250.0,2023,6,6,0,...,0,0,1,0,0,0,0,0,0,0
2,mainpuri,Bewar,Local,800.0,820.0,810.0,2023,6,6,0,...,1,0,0,0,0,0,0,0,0,0
3,chittorgarh,Nimbahera,Other,2040.0,2668.0,2300.0,2023,6,6,0,...,0,0,0,1,0,0,0,0,0,0
4,pratapgarh,Pratapgarh,Other,476.0,1043.0,617.0,2023,6,6,0,...,0,0,0,0,0,0,0,0,0,0


--- LABEL ENCODING ---


In [ ]:

col = ['District Name' , 'Market Name' , 'Variety']


for x in col : 
    le = LabelEncoder() 
    df[x] = le.fit_transform(df[x])

--- TRAIN TEST SPLIT ---

In [ ]:
x = df.drop("Modal_Price" , axis=1)
y = df["Modal_Price"]

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.33, random_state=42)

--- STANDARD SCALER ---

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaler = scaler.fit_transform(x_train)
x_test_scaler = scaler.transform(x_test)

--- MODEL TRAINING ---

In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
# Model
model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function='RMSE',
    eval_metric='R2',
    random_seed=42,
    verbose=100
)

In [ ]:
# Training
model.fit(
    x_train,
    y_train,
    eval_set=(x_test, y_test),
    use_best_model=True
)

0:	learn: 0.0816411	test: 0.0742682	best: 0.0742682 (0)	total: 28.7ms	remaining: 28.7s
100:	learn: 0.9711129	test: 0.9454191	best: 0.9454191 (100)	total: 2.47s	remaining: 22s
200:	learn: 0.9845582	test: 0.9647043	best: 0.9647043 (200)	total: 4.92s	remaining: 19.6s
300:	learn: 0.9890189	test: 0.9684466	best: 0.9684466 (300)	total: 7.39s	remaining: 17.2s
400:	learn: 0.9908752	test: 0.9697588	best: 0.9697588 (400)	total: 9.81s	remaining: 14.7s
500:	learn: 0.9919867	test: 0.9706423	best: 0.9706423 (500)	total: 12.2s	remaining: 12.2s
600:	learn: 0.9926844	test: 0.9710981	best: 0.9710981 (600)	total: 14.6s	remaining: 9.72s
700:	learn: 0.9933091	test: 0.9714013	best: 0.9714013 (700)	total: 17s	remaining: 7.26s
800:	learn: 0.9938500	test: 0.9717177	best: 0.9717192 (799)	total: 19.5s	remaining: 4.84s
900:	learn: 0.9942272	test: 0.9719077	best: 0.9719077 (900)	total: 21.9s	remaining: 2.4s
999:	learn: 0.9945293	test: 0.9720422	best: 0.9720422 (999)	total: 24.3s	remaining: 0us

bestTest = 0.972042

CatBoostRegressor(depth=8, eval_metric='R2', iterations=1000, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [ ]:
# Prediction
y_pred = model.predict(x_test)

# Evaluation
print("R2 Score :", r2_score(y_test, y_pred))
print("MAE :", mean_absolute_error(y_test, y_pred))

R2 Score : 0.9720421608075805
MAE : 58.83985014077153
